# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"Dataset Name: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets in the dataset:")
for record_set in record_sets:
    print(f"- Record set: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - Field @id: {field_id}")
    print("")
# For demonstration, get the list of IDs
record_set_ids = [rs['@id'] for rs in record_sets]
# We'll use the first record set for further steps
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"Primary record set ID: {main_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0 and isinstance(records[0], dict):
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from {record_set_id}")
        else:
            print(f"No tabular data loaded from {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

if main_record_set_id in dataframes:
    print(f"Columns in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set DataFrame available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field relevant for mental health - e.g., 'phq9_score' (Patient Health Questionnaire-9)
# Use the correct @id from the fields overview; replace with actual field id if different
numeric_field = 'phq9_score'  # Replace with correct field/key from your DataFrame

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field in df.columns:
    threshold = 4
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field, such as 'gender' (replace with actual column name/@id from dataset)
    group_field = 'gender'  # Use @id if column is named with the id, else update as needed
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Average '{numeric_field}' grouped by {group_field}:")
        display(grouped_df)
    else:
        print(f"Group field '{group_field}' not found in DataFrame columns: {df.columns.tolist()}")
else:
    print(f"Numeric field '{numeric_field}' not found in columns: {df.columns.tolist() if df is not None else None}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of PHQ-9 scores
if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field} (PHQ-9)")
    plt.xlabel("Score")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by gender
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} Score by {group_field}")
        plt.xlabel(group_field.capitalize())
        plt.ylabel(f"{numeric_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to explore and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We loaded the dataset using its Croissant schema, listed available record sets and fields, extracted the main tabular data, and performed basic exploratory data analysis.

We filtered participant records based on PHQ-9 depression scores, normalized the scores, and grouped them by gender to reveal trends in self-reported mental health by demographic. The visualizations provided further insights into the distribution and differences between groups.

This approach can be extended to further explore other fields (such as GAD-7 anxiety or PSQ stress scores) and demographic factors to inform mental health research and public health interventions in the region.